# Isoglot 07: capability check, before and after LISA

Did the safety fine-tune hurt general ability in Hausa, Yoruba and Igbo? 200 Belebele reading
questions and 200 AfriMMLU knowledge questions per language (English, Hausa, Yoruba, Igbo), the
same draw for both models, scored by log-likelihood through AuditKIT (`MCQAdapter`,
`mcq_loglikelihood`, metric `acc`). No judge, no API cost. The change is paired, question by
question, with a bootstrap 95% interval.

**Before running:** Accelerator **GPU T4 x2** (one is used), Internet **on**, Secret **`HF_TOKEN`**
ticked. **Save Version -> Save & Run All (Commit)**. About 40-60 minutes.


In [ ]:
%%capture
!pip install -q "isoglot[gpu] @ git+https://github.com/nftkingiii/isoglot"


In [ ]:
import os, json, time, gc, torch
from pathlib import Path
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
from huggingface_hub import HfApi, whoami
api = HfApi()
REPO = f"{whoami()['name']}/isoglot-runs"
OUT = Path("/kaggle/working/capability"); OUT.mkdir(parents=True, exist_ok=True)

from isoglot.capability import load_samples
samples = load_samples(n=200, seed=0)
print(len(samples), "questions |", torch.cuda.get_device_name(0))


In [ ]:
from isoglot.capability import score

results = {}
for name, spec in [("base", "hf:CohereLabs/tiny-aya-global"), ("lisa", "hf:NFTKING/isoglot-tiny-aya-lisa")]:
    t0 = time.time()
    results[name] = score(samples, spec)
    gc.collect(); torch.cuda.empty_cache()
    acc = sum(results[name].values()) / len(results[name])
    print(f"{name}: {len(results[name])} scored, overall accuracy {acc:.1%}, {(time.time()-t0)/60:.1f} min")


In [ ]:
from isoglot.capability import paired_table, to_markdown

table = paired_table(samples, results["base"], results["lisa"])
md_table = to_markdown(table)
print(md_table)
(OUT / "capability.md").write_text(
    "Belebele and AfriMMLU, 200 questions per benchmark and language, log-likelihood scoring "
    "via AuditKIT; change is paired with a bootstrap 95% CI.\n\n" + md_table, encoding="utf-8")
(OUT / "capability.json").write_text(json.dumps({"table": table, "per_question": results}, indent=1),
                                     encoding="utf-8")
for f in OUT.iterdir():
    api.upload_file(path_or_fileobj=str(f), path_in_repo=f"capability/{f.name}", repo_id=REPO, repo_type="dataset")
print("uploaded ->", REPO, "capability/")
